<a href="https://colab.research.google.com/github/juanepstein99/DI_Bootcamp/blob/main/Week15/Day4/ExercisesXP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice

Completed notebook covering BERT tokenization, sentiment analysis,
custom model inference, named entity recognition, BERT vs. GPT,
and BERT's role in Retrieval-Augmented Generation (RAG).

## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- required section: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- required section: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Install the required libraries in Google Colab.
# The -q flag keeps the installation output short.
%pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer

# Load the standard uncased BERT tokenizer.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# A simple sentence that contains words BERT can split into WordPiece tokens.
sample_sentence = "BERT makes natural language processing easier and more powerful."

print("Sample sentence:")
print(sample_sentence)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Sample sentence:
BERT makes natural language processing easier and more powerful.


In [3]:
# Convert the sentence into the format expected by BERT.
#
# add_special_tokens=True adds:
#   [CLS] at the beginning
#   [SEP] at the end
#
# padding="max_length" pads the sequence to a fixed length.
# truncation=True shortens text if it is longer than max_length.

encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,
    return_attention_mask=True,
    return_tensors="pt"
)

# Convert the numerical IDs back into readable tokens so we can inspect them.
input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("index | token        | id")
print("-------------------------")

for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:")
print(encoding["attention_mask"][0].tolist())

# Show where BERT inserted special tokens such as [CLS], [SEP] and [PAD].
special_positions = [
    (i, tok)
    for i, tok in enumerate(tokens)
    if tok in tokenizer.all_special_tokens
]

print("\nSpecial tokens (index, token):")
print(special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | bert         | 14324
    2 | makes        |  3084
    3 | natural      |  3019
    4 | language     |  2653
    5 | processing   |  6364
    6 | easier       |  6082
    7 | and          |  1998
    8 | more         |  2062
    9 | powerful     |  3928
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Special tokens (index, token):
[(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD

### Exercise 1 reflection

- **How do `[CLS]` and `[SEP]` behave inside the encoder?**  
  `[CLS]` is added at the beginning of the sequence. In many BERT classification tasks, the final hidden representation of this token is used as a summary representation of the complete input. `[SEP]` marks the end of a sentence or separates two text segments when BERT receives a pair of sentences.

- **How does the attention mask hide padded positions from self-attention?**  
  The attention mask contains `1` for real tokens and `0` for padding tokens. During attention calculations, positions marked with `0` are ignored, so the model does not treat `[PAD]` tokens as meaningful text.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- required section: Record the sentence you tested.
- required section: Capture the label plus confidence score and interpret the result.


In [4]:
from transformers import pipeline

# Load a DistilBERT model that has already been fine-tuned
# on the SST-2 sentiment classification dataset.
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "The new checkout experience is fast, simple, and absolutely fantastic!"

prediction = sentiment_pipeline(sentence)

print("Sentence:")
print(sentence)

print("\nPrediction:")
print(prediction)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence:
The new checkout experience is fast, simple, and absolutely fantastic!

Prediction:
[{'label': 'POSITIVE', 'score': 0.9998875856399536}]


### Exercise 2 reflection

- **Does the predicted label match my expectation? Why or why not?**  
  Yes. The sentence uses clearly positive words such as *fast*, *simple*, and *fantastic*, so I expect the model to classify it as **POSITIVE**.

- **How confident is the model and what does the score tell me?**  
  The returned `score` is the model's confidence for the predicted label. A value close to `1.0` means the model is very confident in its prediction, while a value closer to `0.5` would indicate more uncertainty.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(
        self,
        model_name: str = "distilbert-base-uncased-finetuned-sst-2-english",
        max_length: int = 128
    ):
        # Save configuration
        self.model_name = model_name
        self.max_length = max_length

        # Load tokenizer and sequence-classification model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Use GPU when available, otherwise CPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Evaluation mode disables training-specific behaviour such as dropout
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Basic text cleaning:
        # remove extra spaces while preserving the original words.
        clean_text = re.sub(r"\s+", " ", text).strip()

        # Tokenize and convert the text into PyTorch tensors
        encoded = self.tokenizer(
            clean_text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Move every tensor to the selected device
        encoded = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }

        return encoded

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)

        # No gradients are needed because we are only doing inference
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Convert raw logits into probabilities
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

        # Find the index with the highest probability
        predicted_id = int(torch.argmax(probabilities).item())

        # Translate the class ID into a human-readable label
        label = self.model.config.id2label[predicted_id]
        confidence = float(probabilities[predicted_id].item())

        return {
            "text": text,
            "label": label,
            "confidence": confidence
        }

In [6]:
# Instantiate the custom analyzer and test it with several examples.

analyzer = BERTSentimentAnalyzer()

samples = [
    "I loved the service and I would definitely buy here again.",
    "The experience was terrible and I am very disappointed.",
    "The product arrived today and everything seems fine."
]

for text in samples:
    result = analyzer.predict(text)

    print("Text:", result["text"])
    print("Label:", result["label"])
    print(f"Confidence: {result['confidence']:.4f}")
    print("-" * 60)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I loved the service and I would definitely buy here again.
Label: POSITIVE
Confidence: 0.9998
------------------------------------------------------------
Text: The experience was terrible and I am very disappointed.
Label: NEGATIVE
Confidence: 0.9997
------------------------------------------------------------
Text: The product arrived today and everything seems fine.
Label: POSITIVE
Confidence: 0.9998
------------------------------------------------------------


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- required section: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- required section: Explain how you handled subword tokens that begin with `##`.


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # Load tokenizer and NER model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # Select GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

        # Mapping from numeric prediction IDs to BIO entity labels
        self.id2label = self.model.config.id2label

    def recognize(self, text: str):
        # Tokenize the text and keep offsets so predictions can be mapped
        # back to the original characters.
        encoded = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True
        )

        offset_mapping = encoded.pop("offset_mapping")[0].tolist()

        model_inputs = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            outputs = self.model(**model_inputs)

        predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()
        input_ids = model_inputs["input_ids"][0].cpu().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids)

        entities = []
        current_entity = None

        for token, pred_id, (start, end) in zip(tokens, predictions, offset_mapping):
            label = self.id2label[pred_id]

            # Special tokens such as [CLS] and [SEP] have offset (0, 0)
            if start == end:
                continue

            if label == "O":
                # Close the current entity when we return to an outside token
                if current_entity is not None:
                    entities.append(current_entity)
                    current_entity = None
                continue

            # BIO labels look like B-PER, I-PER, B-ORG, etc.
            prefix, entity_type = label.split("-", 1)

            if prefix == "B" or current_entity is None or current_entity["entity"] != entity_type:
                if current_entity is not None:
                    entities.append(current_entity)

                current_entity = {
                    "entity": entity_type,
                    "start": start,
                    "end": end,
                    "text": text[start:end]
                }

            else:
                # Extend an I- tag over the next word piece/token
                current_entity["end"] = end
                current_entity["text"] = text[current_entity["start"]:end]

        # Add the final open entity
        if current_entity is not None:
            entities.append(current_entity)

        return entities

In [8]:
# Test the recognizer using text containing people, organizations and locations.

ner = BERTNamedEntityRecognizer()

sample_text = (
    "Sundar Pichai works at Google and visited London "
    "before meeting researchers from Microsoft."
)

entities = ner.recognize(sample_text)

print("Text:")
print(sample_text)

print("\nRecognized entities:")
for entity in entities:
    print(
        f"{entity['text']:<20} "
        f"-> {entity['entity']}"
    )

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text:
Sundar Pichai works at Google and visited London before meeting researchers from Microsoft.

Recognized entities:
Sundar Pichai        -> PER
Google               -> ORG
London               -> LOC
Microsoft            -> ORG


## Exercise 5 - Comparing BERT and GPT

**Objective:** Summarize how encoder-style models differ from decoder-style models.

| Category | BERT | GPT |
|----------|------|-----|
| **Architecture** | Transformer **encoder-only** architecture using bidirectional self-attention. | Transformer **decoder-only** architecture using causal/masked self-attention. |
| **Primary purpose** | Understanding and representing text by using context from both directions. | Generating text by predicting the next token from previous tokens. |
| **Typical use cases** | Classification, sentiment analysis, NER, semantic search, embeddings and extractive QA. | Text generation, conversation, summarization, drafting, coding and generative QA. |
| **Strengths** | Strong contextual representations and excellent performance on text-understanding tasks. | Strong natural-language generation, flexible prompting and broad generative capabilities. |
| **Weaknesses** | Not naturally designed for long-form free-text generation. | Autoregressive generation can be computationally expensive and may generate unsupported information. |

### Reflection

BERT and GPT both use the Transformer architecture, but they are optimized for different goals.

BERT reads the available context **bidirectionally**, making it especially useful when the main task is to understand, classify or represent existing text.

GPT processes text **autoregressively**, predicting the next token from the tokens that came before it. This makes GPT naturally suited to generating new text.

So, for a task such as named entity recognition or semantic retrieval, BERT-style models are a strong fit. For generating an explanation, conversation or article, GPT-style models are generally the more natural choice.

## Exercise 6 - BERT inside Retrieval-Augmented Generation

**Objective:** Explain how BERT-generated embeddings can support the retrieval stage of a RAG workflow.

### 1. How BERT encodes queries and documents

A BERT-based encoder converts text into numerical vector representations called **embeddings**. Documents can be divided into smaller chunks and encoded in advance, while the user's query is encoded when the search happens. The goal is for texts with similar meanings to have vector representations that are close to one another in the embedding space.

### 2. How embeddings are stored and searched in a vector database

The document embeddings are stored in a **vector database** together with information about the original document or passage. When a user submits a query, its embedding is compared with the stored vectors using a similarity measure such as cosine similarity or another nearest-neighbour search method. The database returns the document chunks whose embeddings are most similar to the query embedding.

### 3. How retrieved passages are passed to a generative model

After retrieval, the most relevant passages are inserted into the prompt or context supplied to a generative model such as GPT. The generative model then uses both the user's question and the retrieved information to create its answer. This is the core idea of **Retrieval-Augmented Generation (RAG)**: retrieval provides relevant external knowledge, while the generative model turns that information into a useful natural-language response.

### 4. Concrete application example

Imagine a company has thousands of internal support documents. A BERT-based retrieval system can encode those documents and store their embeddings in a vector database. When an employee asks, *"How do I reset a customer's subscription after a failed payment?"*, the query is encoded and matched with the most relevant support documentation. Those passages are then given to GPT, which produces a clear answer grounded in the company's own documentation.

### Final takeaway

BERT and GPT can complement each other in a RAG system:

**BERT-style encoder → retrieve relevant information → vector database → relevant passages → GPT-style generator → final answer**

BERT helps the system **find** useful information, while GPT helps the system **explain and generate** the final response.